# Khai bao tai khoan

In [ ]:
# login = 3928 # XS
# password = '!Od52b03'
# server = 'XSFintech-DEMO' 

login = 6338576 # OANDa
password = '123456Qaz1!'
server = 'OANDA-Demo-1'    

# Lay cac lenh hien tai

In [ ]:
# 

# Lay cac lenh trong qua khu

In [ ]:
import MetaTrader5 as mt5
from datetime import datetime

# Khởi tạo kết nối với MetaTrader 5
if not mt5.initialize(login=login, password=password, server=server):
    print("Initialize() failed, error code =", mt5.last_error())
    quit()
else:
    print("Kết nối thành công với MT5")

    # Lấy tổng số lệnh lịch sử từ tài khoản
    from_date=datetime(2000,1,1)
    to_date=datetime.now()
    history_orders=mt5.history_orders_total(from_date, datetime.now())

    if history_orders is None:
        print("Không có lệnh lịch sử nào trong tài khoản hoặc không thể lấy thông tin lệnh lịch sử.")
    else:
        print(f"Tổng số lệnh lịch sử là: {history_orders}")

    # Ngắt kết nối với MetaTrader 5
    mt5.shutdown()


In [ ]:
import pandas as pd

history_orders = {}
# Khởi tạo kết nối với MetaTrader 5
if not mt5.initialize(login=login, password=password, server=server):
    print("Initialize() failed, error code =", mt5.last_error())
    quit()
else:
    from_date=datetime(2020,1,1)
    to_date=datetime.now()
    history_orders=mt5.history_orders_get(from_date, to_date, symbol="USDJPY")

    print(history_orders)
    print(type(history_orders))

    # Ngắt kết nối với MetaTrader 5
    mt5.shutdown()

# Trích xuất thông tin và tạo list của dictionaries
orders_list = [order._asdict() for order in history_orders]

# orders_list = [
#     {
#         'ticket': order.ticket,
#         'time_setup': order.time_setup,
#         'time_setup_msc': order.time_setup_msc,
#         'time_done': order.time_done,
#         'time_done_msc': order.time_done_msc,
#         'time_expiration': order.time_expiration,
#         'type': order.type,
#         # Thêm tất cả các trường cần thiết ở đây...
#     }
#     for order in history_orders
# ]

# Chuyển list này thành DataFrame
orders_df = pd.DataFrame(orders_list)

# Chuyển đổi các cột thời gian từ timestamp sang datetime (nếu cần)
orders_df['time_setup'] = pd.to_datetime(orders_df['time_setup'], unit='s')
orders_df['time_done'] = pd.to_datetime(orders_df['time_done'], unit='s')
# Và cứ tiếp tục chuyển đổi cho các cột thời gian khác

# In DataFrame để kiểm tra
print(orders_df)
orders_df.to_csv('orders_df.csv')

In [ ]:
orders_df

# Dat lenh buy

In [ ]:
# login = 3928 # XS
# password = '!Od52b03'
# server = 'XSFintech-DEMO'

login = 6338576 # OANDa
password = '123456Qaz1!'
server = 'OANDA-Demo-1'    

In [ ]:
# Hàm để đặt một lệnh mua
import MetaTrader5 as mt5
import math
  
if not mt5.initialize(login=login, password=password, server=server): # Kiem tra san xem co tin hieu
    print("Initialize() failed, error code =", mt5.last_error())
    quit()

else:
    symbol = 'EURUSD.sml'
    lot = 0.01  # Số lượng lô mà bạn muốn mua

    if not mt5.symbol_select(symbol, True): # 
        print(f"Failed to select {symbol}, error code =", mt5.last_error())
        quit()

    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None:
        print(f"{symbol} not found")        

    point = symbol_info.point
    price = mt5.symbol_info_tick(symbol).ask
    deviation = 20  # Độ lệch giá cho phép

    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": mt5.ORDER_TYPE_BUY,
        "price": price,
        # "sl": price - 4000 * point,  # Lệnh dừng lỗ (Stop Loss)
        # "tp": price + 2000 * point,  # Lệnh chốt lời (Take Profit)
		"sl": price - 0.01,  # Lệnh dừng lỗ (Stop Loss)
        "tp": price + 0.02,  # Lệnh chốt lời (Take Profit)
        "deviation": deviation,
        "magic": 23456789,
        "comment": "Autotrading K10",
        "type_time": mt5.ORDER_TIME_GTC,
        # "type_filling": mt5.ORDER_FILLING_IOC,
        "type_filling": mt5.ORDER_FILLING_FOK,        
    }

    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        print("Failed to send order :", result.retcode, result._asdict())
    else:
        print("Order placed BUY successfully!")
        print(result)

    # Đóng kết nối với MT5
    mt5.shutdown()

# Dat lenh sell

In [ ]:
# Hàm để đặt một lệnh bán
import MetaTrader5 as mt5
  
if not mt5.initialize(login=login, password=password, server=server):
    print("Initialize() failed, error code =", mt5.last_error())
    quit()
else:
    lot = 0.01  # Số lượng lô mà bạn muốn mua
    if not mt5.symbol_select(symbol, True): # 
        print(f"Failed to select {symbol}, error code =", mt5.last_error())
        quit()

    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None:
        print(f"{symbol} not found")           
   
    point = symbol_info.point
    price = mt5.symbol_info_tick(symbol).bid  # Sử dụng giá bid cho lệnh bán
    deviation = 20  # Độ lệch giá cho phép

    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": mt5.ORDER_TYPE_SELL,
        "price": price,
        "sl": price + 0.02,  # Lệnh dừng lỗ (Stop Loss) cho lệnh bán
        "tp": price - 0.01,  # Lệnh chốt lời (Take Profit) cho lệnh bán
        "deviation": deviation,
        "magic": 234000,
        "comment": "ABC",
        "type_time": mt5.ORDER_TIME_GTC,
        # "type_filling": mt5.ORDER_FILLING_IOC,
        "type_filling": mt5.ORDER_FILLING_FOK,
        # ORDER_FILLING_RETURN 
    }

    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        print("Failed to send order :", result.retcode, result._asdict())
    else:
        print("Order placed SELL successfully!")
        print(result)

    # Đóng kết nối với MT5
    mt5.shutdown()